In [84]:
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_lg")

In [87]:
df = pd.read_csv("./datasets/news.csv")
df

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real
...,...,...
9895,Wikileaks Admits To Screwing Up IMMENSELY Wit...,Fake
9896,Trump consults Republican senators on Fed chie...,Real
9897,Trump lawyers say judge lacks jurisdiction for...,Real
9898,WATCH: Right-Wing Pastor Falsely Credits Trum...,Fake


In [88]:
def spacy_preprocess(text):
    doc = nlp(text)
    preprocessed = " ".join(
        [
            str(y.lemma_)
            for y in doc
            if not y.is_stop and not y.is_space and not y.is_punct
        ]
    )

    return preprocessed

In [90]:
df["preprocessed_text"] = df["Text"].apply(spacy_preprocess)
df

,Text,label,preprocessed_text
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,Trump Surrogate BRUTALLY stab pathetic VIDEO s...
1,U.S. conservative leader optimistic of common ...,Real,U.S. conservative leader optimistic common gro...
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,trump propose U.S. tax overhaul stir concern d...
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,Court Forces Ohio allow million illegally purg...
4,Democrats say Trump agrees to work on immigrat...,Real,Democrats Trump agree work immigration bill wa...
...,...,...,...
9895,Wikileaks Admits To Screwing Up IMMENSELY Wit...,Fake,Wikileaks admit screw immensely Twitter Poll H...
9896,Trump consults Republican senators on Fed chie...,Real,trump consult republican senator Fed chief can...
9897,Trump lawyers say judge lacks jurisdiction for...,Real,Trump lawyer judge lack jurisdiction defamatio...
9898,WATCH: Right-Wing Pastor Falsely Credits Trum...,Fake,watch right wing Pastor falsely credit Trump S...


In [91]:
df["spacy_vec"] = df["preprocessed_text"].apply(lambda x: nlp(x).vector)
df

,Text,label,preprocessed_text,spacy_vec
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,Trump Surrogate BRUTALLY stab pathetic VIDEO s...,"[-0.21601638, 0.12463496, -0.02371346, 0.02544..."
1,U.S. conservative leader optimistic of common ...,Real,U.S. conservative leader optimistic common gro...,"[-0.05523487, 0.13486932, 0.013587801, 0.07916..."
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,trump propose U.S. tax overhaul stir concern d...,"[-0.25053078, 0.1549564, 0.07994621, -0.039873..."
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,Court Forces Ohio allow million illegally purg...,"[-0.07829542, 0.08889477, 0.1565298, 0.0183144..."
4,Democrats say Trump agrees to work on immigrat...,Real,Democrats Trump agree work immigration bill wa...,"[-0.10885675, 0.050598655, 0.06811445, 0.05349..."
...,...,...,...,...
9895,Wikileaks Admits To Screwing Up IMMENSELY Wit...,Fake,Wikileaks admit screw immensely Twitter Poll H...,"[-0.13730831, 0.11130498, -0.040160086, 0.0696..."
9896,Trump consults Republican senators on Fed chie...,Real,trump consult republican senator Fed chief can...,"[-0.13811707, 0.14289007, 0.095607586, 0.04331..."
9897,Trump lawyers say judge lacks jurisdiction for...,Real,Trump lawyer judge lack jurisdiction defamatio...,"[-0.16307034, 0.098757535, 0.014880662, -0.020..."
9898,WATCH: Right-Wing Pastor Falsely Credits Trum...,Fake,watch right wing Pastor falsely credit Trump S...,"[-0.12513933, 0.20639922, -0.025848687, -0.039..."


In [92]:
import gensim.downloader as api

wv = api.load("word2vec-google-news-300")

In [94]:
df["gensim_vec"] = df["preprocessed_text"].apply(
    lambda x: wv.get_mean_vector(x.split(" "))
)
df.loc[:, ["spacy_vec", "gensim_vec"]]

,spacy_vec,gensim_vec
0,"[-0.21601638, 0.12463496, -0.02371346, 0.02544...","[0.0085234195, 0.019263458, -0.010577418, 0.03..."
1,"[-0.05523487, 0.13486932, 0.013587801, 0.07916...","[0.00861828, 0.007408227, 0.0007675802, 0.0138..."
2,"[-0.25053078, 0.1549564, 0.07994621, -0.039873...","[0.01793007, 0.006029178, -0.0054984074, 0.038..."
3,"[-0.07829542, 0.08889477, 0.1565298, 0.0183144...","[0.0124946935, 0.0121258395, -0.00019833064, 0..."
4,"[-0.10885675, 0.050598655, 0.06811445, 0.05349...","[-0.002259819, 0.01164962, 0.0036556108, 0.028..."
...,...,...
9895,"[-0.13730831, 0.11130498, -0.040160086, 0.0696...","[0.009134042, 0.01566266, -0.0048167696, 0.031..."
9896,"[-0.13811707, 0.14289007, 0.095607586, 0.04331...","[0.010149229, 0.003485544, -0.003229477, 0.025..."
9897,"[-0.16307034, 0.098757535, 0.014880662, -0.020...","[0.0033575508, 0.0044489778, 0.010112801, -0.0..."
9898,"[-0.12513933, 0.20639922, -0.025848687, -0.039...","[0.0055006472, 0.015122254, 0.00051397725, 0.0..."


In [96]:
print(df["label"].value_counts())

label
Fake    5000
Real    4900
Name: count, dtype: int64


In [100]:
df["label_num"] = df["label"].map({"Fake": 0, "Real": 1})

In [101]:
import numpy as np
from sklearn.model_selection import train_test_split

X = df["spacy_vec"]
y = df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train = np.stack(X_train)
X_test = np.stack(X_test)

In [102]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [103]:
import keras

model = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        keras.layers.Flatten(),
        keras.layers.Dense(100, activation="relu"),
        keras.layers.Dense(50, activation="relu"),
        keras.layers.Dense(3, activation="softmax"),
    ]
)
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 3)              │           153 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,303 (137.90 KB)

 Trainable params: 35,303 (137.90 KB)

 Non-trainable params: 0 (0.00 B)

In [104]:
model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

In [105]:
model.fit(X_train, y_train, epochs=10)

Epoch 1/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 514us/step - accuracy: 0.9038 - loss: 0.2518  
Epoch 2/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 533us/step - accuracy: 0.9664 - loss: 0.0900
Epoch 3/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 526us/step - accuracy: 0.9795 - loss: 0.0563
Epoch 4/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 498us/step - accuracy: 0.9785 - loss: 0.0573
Epoch 5/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 471us/step - accuracy: 0.9823 - loss: 0.0488
Epoch 6/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 468us/step - accuracy: 0.9850 - loss: 0.0390
Epoch 7/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 472us/step - accuracy: 0.9870 - loss: 0.0315
Epoch 8/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 462us/step - accuracy: 0.9890 - loss: 0.0298
Epoch 9/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 464us/step - accuracy: 0.9890 - loss: 0.0296
Epoch 10/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 465us/step - accuracy: 0.9886 - loss: 0.0316


In [106]:
model.evaluate(X_test, y_test)

62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9944 - loss: 0.0194 


[0.019429828971624374, 0.9944444298744202]

In [107]:
from sklearn.metrics import classification_report

y_hat_ = model.predict(X_test)
y_hat = np.argmax(y_hat_, axis=1)
print(classification_report(y_test, y_hat))

62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 751us/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1000
           1       0.99      0.99      0.99       980

    accuracy                           0.99      1980
   macro avg       0.99      0.99      0.99      1980
weighted avg       0.99      0.99      0.99      1980



In [108]:
X = df["gensim_vec"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train = np.stack(X_train)
X_test = np.stack(X_test)

In [109]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [110]:
model = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        keras.layers.Flatten(),
        keras.layers.Dense(100, activation="relu"),
        keras.layers.Dense(50, activation="relu"),
        keras.layers.Dense(3, activation="softmax"),
    ]
)
model.summary()


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 3)              │           153 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,303 (137.90 KB)

 Trainable params: 35,303 (137.90 KB)

 Non-trainable params: 0 (0.00 B)

In [111]:
model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

In [112]:
model.fit(X_train, y_train, epochs=10)

Epoch 1/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 1s 479us/step - accuracy: 0.9788 - loss: 0.0612
Epoch 2/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 460us/step - accuracy: 0.9984 - loss: 0.0056
Epoch 3/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 467us/step - accuracy: 0.9999 - loss: 0.0016
Epoch 4/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 463us/step - accuracy: 0.9995 - loss: 0.0015
Epoch 5/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 461us/step - accuracy: 1.0000 - loss: 2.9366e-04
Epoch 6/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 458us/step - accuracy: 1.0000 - loss: 1.0265e-04
Epoch 7/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 462us/step - accuracy: 1.0000 - loss: 7.1125e-05
Epoch 8/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 474us/step - accuracy: 1.0000 - loss: 5.2234e-05
Epoch 9/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 460us/step - accuracy: 1.0000 - loss: 3.9720e-05
Epoch 10/10
248/248 ━━━━━━━━━━━━━━━━━━━━ 0s 452us/step - accuracy: 1.0000 - loss: 3.1589e-05


In [113]:
model.evaluate(X_test, y_test)

62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 490us/step - accuracy: 0.9949 - loss: 0.0164


[0.016369277611374855, 0.9949495196342468]

In [114]:
from sklearn.metrics import classification_report

y_hat_ = model.predict(X_test)
y_hat = np.argmax(y_hat_, axis=1)
print(classification_report(y_test, y_hat))


62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 695us/step
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1000
           1       0.99      1.00      0.99       980

    accuracy                           0.99      1980
   macro avg       0.99      0.99      0.99      1980
weighted avg       0.99      0.99      0.99      1980

